In [1]:
!pip freeze

aiofiles==22.1.0
aiosqlite==0.19.0
alembic==1.11.1
anyio==3.7.0
apprise==1.4.0
argon2-cffi==21.3.0
argon2-cffi-bindings==21.2.0
arrow==1.2.3
asgi-lifespan==2.1.0
asttokens==2.2.1
asyncpg==0.27.0
attrs==23.1.0
Babel==2.12.1
backcall==0.2.0
beautifulsoup4==4.12.2
black==23.3.0
bleach==6.0.0
blinker==1.6.2
boto3==1.26.142
botocore==1.29.142
briefcase==0.3.8
cachetools==5.3.1
certifi==2023.5.7
cffi==1.15.1
charset-normalizer==3.1.0
click==8.1.3
cloudpickle==2.2.1
cmaes==0.9.1
colorama==0.4.6
colorlog==6.7.0
comm==0.1.3
contourpy==1.0.7
cookiecutter==2.1.1
coolname==2.2.0
cramjam==2.6.2
croniter==1.3.15
cryptography==41.0.1
cycler==0.11.0
databricks-cli==0.17.7
dateparser==1.1.8
debugpy==1.6.7
decorator==5.1.1
defusedxml==0.7.1
distlib==0.3.6
docker==6.1.2
entrypoints==0.4
exceptiongroup==1.1.1
executing==1.2.0
fastapi==0.97.0
fastjsonschema==2.17.1
fastparquet==2023.4.0
filelock==3.12.2
Flask==2.3.2
fonttools==4.39.4
fqdn==1.5.1
fsspec==2023.5.0
future==0.18.3
gitdb==4.0.10
GitPython==3.1.

In [2]:
import pickle
import pandas as pd

In [3]:
year = 2022
month = 2

input_file = f'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{year:04d}-{month:02d}.parquet'
output_file = f'output/yellow_tripdata_{year:04d}-{month:02d}.parquet'

In [4]:
with open('model.bin', 'rb') as f_in:
    dv, model = pickle.load(f_in)

In [5]:
categorical = ['PULocationID', 'DOLocationID']

def read_data(filename):
    df = pd.read_parquet(filename)
    
    df['duration'] = df.tpep_dropoff_datetime - df.tpep_pickup_datetime
    df['duration'] = df.duration.dt.total_seconds() / 60

    df = df[(df.duration >= 1) & (df.duration <= 60)].copy()

    df[categorical] = df[categorical].fillna(-1).astype('int').astype('str')
    
    return df

In [6]:
# df = read_data('https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2022-02.parquet')
df = read_data(input_file)
df['ride_id'] = f'{year:04d}/{month:02d}_' + df.index.astype('str')

In [7]:
dicts = df[categorical].to_dict(orient='records')
X_val = dv.transform(dicts)
y_pred = model.predict(X_val)

In [8]:
y_pred.std()

5.28140357655334

In [9]:
df_result = pd.DataFrame()
df_result['ride_id'] = df['ride_id']
df_result['predicted_duration'] = y_pred

In [10]:
df_result

,ride_id,predicted_duration
0,2022/02_0,18.527783
1,2022/02_1,23.065782
2,2022/02_2,33.686359
3,2022/02_3,23.757436
4,2022/02_4,21.492904
...,...,...
2979426,2022/02_2979426,12.038225
2979427,2022/02_2979427,11.441569
2979428,2022/02_2979428,11.890459
2979429,2022/02_2979429,15.102681


In [11]:
df_result.to_parquet(
    output_file,
    engine='pyarrow',
    compression=None,
    index=False
)

In [12]:
import os
file_stats = os.stat(output_file)
print(f'File Size in Kilobytes is {round(file_stats.st_size / (1024),2)}')

File Size in Kilobytes is 58588.8
